# Melbourne House Price Analysis — Final Solution

This notebook implements two tasks:

**Task 1:** Predict price using selected features:
- Car
- Landsize
- BuildingArea
- YearBuilt (engineered into `HouseAge`)

**Task 2:** Predict price using **all numeric features**.

We use:
- Median imputation for missing values
- Feature engineering (`HouseAge`)
- Linear Regression
- Train/Test split + Cross Validation (R²)

> **Pass Criteria:** Test R² ≥ 0.5


In [6]:
# ===============================
# 1. IMPORTS
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer


## 2. Load Data

In [7]:
# Replace with your dataset path
df = pd.read_csv("melb_data.csv")

# Quick preview
df.head()


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


## 3. Select Required Columns

In [8]:
cols = ['Price', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt']
data = df[cols].copy()

data.head()


,Price,Car,Landsize,BuildingArea,YearBuilt
0,1480000.0,1.0,202.0,NaN,NaN
1,1035000.0,0.0,156.0,79.0,1900.0
2,1465000.0,0.0,134.0,150.0,1900.0
3,850000.0,1.0,94.0,NaN,NaN
4,1600000.0,2.0,120.0,142.0,2014.0


## 4. Handle Missing Values (Median Imputation)

In [9]:
imputer = SimpleImputer(strategy='median')

data[['Car', 'Landsize', 'BuildingArea', 'YearBuilt']] = imputer.fit_transform(
    data[['Car', 'Landsize', 'BuildingArea', 'YearBuilt']]
)

data.isnull().sum()


Price           0
Car             0
Landsize        0
BuildingArea    0
YearBuilt       0
dtype: int64

## 5. Feature Engineering

In [10]:
# Convert YearBuilt → HouseAge
data['HouseAge'] = 2025 - data['YearBuilt']

# Drop original YearBuilt
data.drop('YearBuilt', axis=1, inplace=True)

data.head()


,Price,Car,Landsize,BuildingArea,HouseAge
0,1480000.0,1.0,202.0,126.0,55.0
1,1035000.0,0.0,156.0,79.0,125.0
2,1465000.0,0.0,134.0,150.0,125.0
3,850000.0,1.0,94.0,126.0,55.0
4,1600000.0,2.0,120.0,142.0,11.0


## 6. Task 1: Selected Features Model

In [11]:
X_task1 = data[['Car', 'Landsize', 'BuildingArea', 'HouseAge']]
y = data['Price']

x_train, x_test, y_train, y_test = train_test_split(
    X_task1, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(x_train, y_train)

trainScore = model.score(x_train, y_train)
testScore = model.score(x_test, y_test)

print("==== TASK 1 (Given Features) ====")
print(f"Train Score: {trainScore:.3f}")
print(f"Test Score : {testScore:.3f}")

if testScore >= 0.5:
    print("PASS (>= 0.5)")
else:
    print("Needs Improvement")


==== TASK 1 (Given Features) ====
Train Score: 0.139
Test Score : 0.132
Needs Improvement


## 7. Cross Validation

In [12]:
cv_scores = cross_val_score(model, X_task1, y, cv=5, scoring='r2')

print("Cross Validation Scores:", cv_scores)
print("Mean CV Score:", round(cv_scores.mean(), 3))


Cross Validation Scores: [ 0.1001914   0.19825156  0.15077828  0.07129647 -4.43030759]
Mean CV Score: -0.782


## 8. Task 2: All Features Model

In [13]:
# Select all numeric features
all_features = df.select_dtypes(include=[np.number]).copy()

# Handle missing values
all_features = pd.DataFrame(
    SimpleImputer(strategy='median').fit_transform(all_features),
    columns=all_features.columns
)

y_all = all_features['Price']
X_all = all_features.drop('Price', axis=1)

x_train, x_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

model2 = LinearRegression()
model2.fit(x_train, y_train)

trainScore2 = model2.score(x_train, y_train)
testScore2 = model2.score(x_test, y_test)

print("==== TASK 2 (All Features) ====")
print(f"Train Score: {trainScore2:.3f}")
print(f"Test Score : {testScore2:.3f}")

if testScore2 >= 0.5:
    print("PASS (>= 0.5)")
else:
    print("Needs Improvement")


==== TASK 2 (All Features) ====
Train Score: 0.518
Test Score : 0.519
PASS (>= 0.5)


## 9. Final Comparison

In [14]:
print("==== FINAL COMPARISON ====")
print(f"Task 1 Test Score: {testScore:.3f}")
print(f"Task 2 Test Score: {testScore2:.3f}")


==== FINAL COMPARISON ====
Task 1 Test Score: 0.132
Task 2 Test Score: 0.519
